
# Machine Learning Modeling Pipeline for Materials Property Prediction

This notebook compares:
- Basic ML models

Targets:
- Formation Energy
- Energy Above Hull
- Band Gap

Metrics:
- MAE, RMSE, R²


In [ ]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor

import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns

from crabnet.model import CrabNet
from crabnet.utils.utils import set_seeds



ModuleNotFoundError: No module named 'CrabNet'

In [ ]:
!pip uninstall -y crabnet


Found existing installation: crabnet 1.0.1
Uninstalling crabnet-1.0.1:
  Successfully uninstalled crabnet-1.0.1


In [ ]:
!pip install torch --index-url https://download.pytorch.org/whl/cpu
!pip install pymatgen


Looking in indexes: https://download.pytorch.org/whl/cpu


In [ ]:
# ===============================
# Load Gold-Standard Dataset
# ===============================

import pandas as pd

DATA_PATH = "regression_gold.csv"   # cleaned & reduced dataset
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully ✅")
print("Shape:", df.shape)
df.head()


Dataset loaded successfully ✅
Shape: (17475, 15)


,formula_pretty,avg_atomic_number,avg_electronegativity,valence_electron_count,fraction_transition_metals,nelements,density,volume,nsites,crystal_system,spacegroup_number,point_group,formation_energy_per_atom,energy_above_hull,band_gap
0,Rb2U(PtSe2)3,50.333333,2.096667,10.916667,0.250000,4,6.998436,696.621988,24,Orthorhombic,69,mmm,-1.085894,0.000000,0.0
1,V4C3,15.714286,2.024286,8.857143,0.571429,2,5.730004,69.492817,7,Trigonal,160,3m,-0.495103,0.119647,0.0
2,PaTiTc2,49.750000,1.710000,5.250000,0.750000,3,11.306843,69.744911,4,Cubic,225,m-3m,-0.361549,0.000000,0.0
3,Ti4Al11Ni,16.187500,1.611250,10.562500,0.312500,3,3.650527,248.798536,16,Tetragonal,99,4mm,-0.395257,0.052130,0.0
4,Mg(CuO2)4,14.769231,2.802308,13.384615,0.307692,3,4.647838,145.225133,13,Cubic,216,-43m,-1.018523,0.113782,0.0


In [ ]:

NUM_FEATURES = [
    "avg_atomic_number",
    "avg_electronegativity",
    "valence_electron_count",
    "fraction_transition_metals",
    "nelements",
    "density",
    "volume",
    "nsites"
]

CAT_FEATURES = ["crystal_system"]

TARGETS = {
    "Formation Energy": "formation_energy_per_atom",
    "Energy Above Hull": "energy_above_hull",
    "Band Gap": "band_gap"
}


In [ ]:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [ ]:

preprocessor = ColumnTransformer(
    [
        ("num", StandardScaler(), NUM_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATURES)
    ]
)


In [ ]:

def regression_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mse),
        "R2": r2_score(y_true, y_pred)
    }


In [ ]:
# ===============================
# ML model imports
# ===============================

from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor

import xgboost as xgb


In [ ]:

# Basic ML models
ml_models = {
    "Random Forest": RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42),
    "SVR": SVR(C=5, epsilon=0.05),
    "KNN": KNeighborsRegressor(n_neighbors=5),
    "MLP": MLPRegressor(hidden_layer_sizes=(64,32), max_iter=200, early_stopping=True, random_state=42),
    "XGBoost": xgb.XGBRegressor(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.08,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        objective="reg:squarederror",
        random_state=42
    )
}


In [ ]:

from sklearn.model_selection import train_test_split
# Train-test split
indices = np.arange(len(df))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)

X_tab = df[NUM_FEATURES + CAT_FEATURES]
Xtr_tab = preprocessor.fit_transform(X_tab.iloc[train_idx])
Xte_tab = preprocessor.transform(X_tab.iloc[test_idx])

if hasattr(Xtr_tab, "toarray"):
    Xtr_tab = Xtr_tab.toarray()
    Xte_tab = Xte_tab.toarray()


In [ ]:
# ===============================
# Regression metrics imports
# ===============================

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


In [ ]:
def regression_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mse),
        "R2": r2_score(y_true, y_pred)
    }


In [ ]:
results = []

for task, target in TARGETS.items():
    y = df[target].values
    ytr, yte = y[train_idx], y[test_idx]

    for name, model in ml_models.items():
        model.fit(Xtr_tab, ytr)
        preds = model.predict(Xte_tab)

        res = regression_metrics(yte, preds)
        res.update({
            "Model": name,
            "Target": task
        })
        results.append(res)


In [ ]:
# ===============================
# Display ML model results
# ===============================

results_df = pd.DataFrame(results)
results_df


,MAE,RMSE,R2,Model,Target
0,0.297862,0.442212,0.832867,Random Forest,Formation Energy
1,0.391460,0.561087,0.730932,SVR,Formation Energy
2,0.418652,0.617181,0.674443,KNN,Formation Energy
3,0.400645,0.548935,0.742461,MLP,Formation Energy
4,0.364021,0.499800,0.786502,XGBoost,Formation Energy
5,0.065791,0.137942,0.294793,Random Forest,Energy Above Hull
6,0.069087,0.144741,0.223557,SVR,Energy Above Hull
7,0.069276,0.150782,0.157391,KNN,Energy Above Hull
8,0.074922,0.143320,0.238728,MLP,Energy Above Hull
9,0.068405,0.139150,0.282385,XGBoost,Energy Above Hull


In [ ]:
!pip uninstall -y crabnet
!pip install crabnet-kingcrab


ERROR: Could not find a version that satisfies the requirement crabnet-kingcrab (from versions: none)
ERROR: No matching distribution found for crabnet-kingcrab


In [ ]:
import pandas as pd
import numpy as np

from pymatgen.core import Composition, Element

import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:
DATA_PATH = "regression_gold.csv"   # your dataset
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()


Dataset shape: (17475, 15)


,formula_pretty,avg_atomic_number,avg_electronegativity,valence_electron_count,fraction_transition_metals,nelements,density,volume,nsites,crystal_system,spacegroup_number,point_group,formation_energy_per_atom,energy_above_hull,band_gap
0,Rb2U(PtSe2)3,50.333333,2.096667,10.916667,0.250000,4,6.998436,696.621988,24,Orthorhombic,69,mmm,-1.085894,0.000000,0.0
1,V4C3,15.714286,2.024286,8.857143,0.571429,2,5.730004,69.492817,7,Trigonal,160,3m,-0.495103,0.119647,0.0
2,PaTiTc2,49.750000,1.710000,5.250000,0.750000,3,11.306843,69.744911,4,Cubic,225,m-3m,-0.361549,0.000000,0.0
3,Ti4Al11Ni,16.187500,1.611250,10.562500,0.312500,3,3.650527,248.798536,16,Tetragonal,99,4mm,-0.395257,0.052130,0.0
4,Mg(CuO2)4,14.769231,2.802308,13.384615,0.307692,3,4.647838,145.225133,13,Cubic,216,-43m,-1.018523,0.113782,0.0


In [ ]:
def crabnet_style_encoder(formula):
    """
    Convert chemical formula into composition-aware numerical features
    """
    comp = Composition(formula).fractional_composition

    Z, EN, TM, GROUP = [], [], [], []

    for el, frac in comp.items():
        Z.append(frac * el.Z)                           # atomic number
        EN.append(frac * (el.X if el.X else 0))         # electronegativity
        TM.append(frac * int(el.is_transition_metal))   # transition metal fraction
        GROUP.append(frac * (el.group if el.group else 0))  # valence proxy

    return [
        sum(Z),          # avg atomic number
        sum(EN),         # avg electronegativity
        sum(TM),         # fraction of transition metals
        sum(GROUP),      # valence proxy
        len(comp)        # number of elements
    ]


In [ ]:
X_comp = df["formula_pretty"].apply(crabnet_style_encoder)

X_comp = pd.DataFrame(
    X_comp.tolist(),
    columns=[
        "avg_atomic_number",
        "avg_electronegativity",
        "fraction_transition_metals",
        "valence_proxy",
        "nelements"
    ]
)

X_comp.head()


,avg_atomic_number,avg_electronegativity,fraction_transition_metals,valence_proxy,nelements
0,50.333333,2.096667,0.250000,10.916667,4
1,15.714286,2.024286,0.571429,8.857143,2
2,49.750000,1.710000,0.750000,5.250000,3
3,16.187500,1.611250,0.312500,10.562500,3
4,14.769231,2.802308,0.307692,13.384615,3


In [ ]:
class CrabNetLite(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
def train_and_evaluate(X, y, epochs=100):
    # Train-test split
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Convert to tensors
    Xtr = torch.tensor(Xtr.values, dtype=torch.float32)
    Xte = torch.tensor(Xte.values, dtype=torch.float32)
    ytr = torch.tensor(ytr.values, dtype=torch.float32).view(-1, 1)

    # Model
    model = CrabNetLite(Xtr.shape[1])
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.L1Loss()   # MAE loss (standard for materials ML)

    # Training loop
    for _ in range(epochs):
        optimizer.zero_grad()
        loss = loss_fn(model(Xtr), ytr)
        loss.backward()
        optimizer.step()

    # Prediction
    preds = model(Xte).detach().numpy().flatten()

    # Metrics
    return {
        "MAE": mean_absolute_error(yte, preds),
        "RMSE": np.sqrt(mean_squared_error(yte, preds)),
        "R2": r2_score(yte, preds)
    }


In [ ]:
TARGETS = {
    "Band Gap": "band_gap",
    "Formation Energy": "formation_energy_per_atom",
    "Energy Above Hull": "energy_above_hull"
}

results = []

for name, col in TARGETS.items():
    metrics = train_and_evaluate(X_comp, df[col], epochs=100)
    metrics["Target"] = name
    metrics["Model"] = "CrabNet-Style Composition DL"
    results.append(metrics)

results_df = pd.DataFrame(results)
results_df


,MAE,RMSE,R2,Target,Model
0,1.365314,1.721105,0.214936,Band Gap,CrabNet-Style Composition DL
1,0.690855,0.903104,0.302929,Formation Energy,CrabNet-Style Composition DL
2,0.068725,0.173392,-0.114260,Energy Above Hull,CrabNet-Style Composition DL


In [ ]:
results_df.to_csv("crabnet_style_results.csv", index=False)


Hybrid model

In [ ]:
# ===============================
# Hybrid feature matrix
# ===============================

X_hybrid = pd.concat(
    [
        X_tab.reset_index(drop=True),   # your existing tabular features
        X_comp.reset_index(drop=True)   # CrabNet-style composition embeddings
    ],
    axis=1
)

Xtr_hybrid = X_hybrid.iloc[train_idx]
Xte_hybrid = X_hybrid.iloc[test_idx]


In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer


In [ ]:
# Identify column types
cat_cols = X_hybrid.select_dtypes(include=["object"]).columns.tolist()
num_cols = X_hybrid.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical columns:", cat_cols)
print("Numerical columns:", num_cols)


Categorical columns: ['crystal_system']
Numerical columns: ['avg_atomic_number', 'avg_electronegativity', 'valence_electron_count', 'fraction_transition_metals', 'nelements', 'density', 'volume', 'nsites', 'avg_atomic_number', 'avg_electronegativity', 'fraction_transition_metals', 'valence_proxy', 'nelements']


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
    ]
)


In [ ]:
Xtr_enc = preprocessor.fit_transform(X_hybrid.iloc[train_idx])
Xte_enc = preprocessor.transform(X_hybrid.iloc[test_idx])

print(Xtr_enc.dtype)
print(Xtr_enc.shape)


ValueError: Selected columns, ['avg_atomic_number', 'avg_electronegativity', 'valence_electron_count', 'fraction_transition_metals', 'nelements', 'density', 'volume', 'nsites', 'avg_atomic_number', 'avg_electronegativity', 'fraction_transition_metals', 'valence_proxy', 'nelements'], are not unique in dataframe

In [ ]:
import xgboost as xgb

xgb_model = xgb.XGBRegressor(
    n_estimators=800,
    max_depth=6,
    learning_rate=0.04,
    subsample=0.85,
    colsample_bytree=0.85,
    objective="reg:squarederror",
    tree_method="hist",
    random_state=42
)

xgb_model.fit(Xtr_hybrid, ytr)
xgb_preds = xgb_model.predict(Xte_hybrid)

print("XGBoost baseline:", regression_metrics(yte, xgb_preds))


ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:crystal_system: object